# Introducción

Este proyecto utiliza la **API-FOOTBALL v3**, una API REST con información actualizada e histórica de más de **1100 ligas y copas** (resultados, alineaciones, eventos, clasificaciones y estadísticas).  
El acceso se realiza mediante solicitudes **HTTP autenticadas con API key**, con filtros por **fecha, liga, equipo o temporada**.

En este notebook se documenta y ejecuta el **pipeline ETL orquestado con Prefect**, cuya lógica reside en `scripts/etl_fixtures.py`.  
Se muestra el **setup**, la **ejecución del flow**, y cómo **servir y monitorear** el flujo desde **Prefect Cloud**.

El pipeline persiste datos en un **Data Lake estructurado en capas Delta Lake** (**Bronze → Silver → Gold**) y genera **exportables en CSV y Parquet**.

Además, el flujo puede ejecutarse:
- 🧩 **Manualmente** desde este notebook, para validaciones o cargas controladas.  
- 🔁 **Automáticamente**, programado una vez al día mediante Prefect (cron diario a las 06:00 UTC).


## 1. Uso del flujo desde `scripts/etl_fixtures.py`

El flujo ETL está definido en `scripts/etl_fixtures.py`.  
En este notebook es posible:

- **1.1 Opción A — Corrida única (demo one-off):** ejecutar el flow una sola vez para validar la orquestación.  
- **1.2 Opción B — Servir el flow (opcional):** mantener el flow activo como servicio local (con programación automática).


In [1]:
import prefect
print("Prefect versión:", prefect.__version__)

Prefect versión: 2.20.9


### 1.1 Opción A — Ejecutar una corrida orquestada (demo one-off)


In [ ]:
import importlib

# Importa el script de orquestación
etl = importlib.import_module("scripts.etl_fixtures")

# Ejecuta el flujo ETL de forma local (modo recomendado dentro del Notebook)
etl.etl_parametrizable(endpoints=["fixtures"])

# Nota:
# El mismo flujo también puede ejecutarse desde la terminal:
#     python scripts/etl_fixtures.py
# o dentro del Notebook:
#     !python scripts/etl_fixtures.py
# Todas estas opciones llaman al mismo flow definido en scripts/etl_fixtures.py.

09:53:41.418 | INFO    | prefect.engine - Created flow run 'blue-manatee' for flow 'etl-parametrizable'

09:53:41.421 | INFO    | Flow run 'blue-manatee' - View at https://app.prefect.cloud/account/1513bf29-3686-40b8-9dbf-c85ba6a6f8c0/workspace/377710aa-6343-48a1-a52d-8fd9be6fbba7/flow-runs/flow-run/0692849d-5239-7b4a-8000-57446d006a74

09:53:42.121 | INFO    | Flow run 'blue-manatee' - Created task run 'extract_data-0' for task 'extract_data'

09:53:42.124 | INFO    | Flow run 'blue-manatee' - Executing 'extract_data-0' immediately...

09:53:44.069 | INFO    | Task run 'get_data-fixtures' - Finished in state Completed()

...

09:53:50.387 | INFO    | Flow run 'blue-manatee' - Created task run 'transform_to_gold_from_silver-0' for task 'transform_to_gold_from_silver'

09:53:50.387 | INFO    | Flow run 'blue-manatee' - Executing 'transform_to_gold_from_silver-0' immediately...

09:53:52.204 | INFO    | Task run 'to_gold-fixtures' - Finished in state Completed()

09:53:52.680 | INFO    | Flow run 'blue-manatee' - Finished in state Completed('All states completed.')

[Completed(message=None, type=COMPLETED, result=UnpersistedResult(type='unpersisted', artifact_type='result', artifact_description='Unpersisted result of type `list`')),
 Completed(message=None, type=COMPLETED, result=UnpersistedResult(type='unpersisted', artifact_type='result', artifact_description='Unpersisted result of type `DataFrame`')),
 Completed(message=None, type=COMPLETED, result=UnpersistedResult(type='unpersisted', artifact_type='result', artifact_description='Unpersisted result of type `NoneType`')),
 Completed(message=None, type=COMPLETED, result=UnpersistedResult(type='unpersisted', artifact_type='result', artifact_description='Unpersisted result of type `DataFrame`')),
 Completed(message=None, type=COMPLETED, result=UnpersistedResult(type='unpersisted', artifact_type='result', artifact_description='Unpersisted result of type `DataFrame`'))]

### 1.2 Opción B — Servir el flow como agente local (opcional)

Esta modalidad permite dejar el flujo **en ejecución continua** y gestionarlo desde **Prefect Cloud** (ejecuciones, logs, programación).

⚠️ Al activarla, la celda quedará ocupada hasta que la detengas manualmente (*Interrupt/Stop*).  
Por defecto permanece **comentada** para evitar ejecuciones involuntarias.

Si se habilita, el flujo se ejecutará **una vez al día (06:00 UTC)** según la configuración en `scripts/etl_fixtures.py`.

In [ ]:
import importlib
etl = importlib.import_module("scripts.etl_fixtures")
# etl.etl_parametrizable.serve(name="ETL-Fixtures", endpoints=["fixtures"])

## 2. Monitoreo en Prefect

El monitoreo del flujo no se realiza desde el notebook, sino desde la **UI de Prefect Cloud**.  
Una vez que el flow se ejecuta (corrida única o servido como agente), es posible:

1. **Flows** → ver el listado de flows registrados (ejemplo: `ETL-Fixtures`).  
2. **Run history** → revisar el historial de ejecuciones con sus logs, tiempos y reintentos.  
3. **Blocks** → administrar la configuración de almacenamiento o infraestructura remota si se utiliza.  
4. **Schedules** → programar ejecuciones automáticas (requiere plan pago).